<a href="https://colab.research.google.com/github/Asmina-hub/LLM-from-scratch/blob/main/positional_embedding.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Data sampling with a sliding window

In [1]:
!pip install tiktoken
!pip install torch
!pip install PyPDF2

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 232.6/232.6 kB 8.3 MB/s eta 0:00:00


In [5]:
import tiktoken
import torch
from PyPDF2 import PdfReader
import numpy as np
from torch.utils.data import Dataset,DataLoader
import torch.nn as nn

In [3]:
with open('harrypotter.pdf','rb') as f:
  reader =PdfReader(f)
  pdf_text = ""
  for i in range(11,len(reader.pages)-1):
    page = reader.pages[i]
    pdf_text = pdf_text + page.extract_text()

  print(f"Total extracted characters: {len(pdf_text)}")
  print("First 500 characters of the extracted text:")
  print(pdf_text[:500])



Total extracted characters: 6279327
First 500 characters of the extracted text:
M
	
CHAPTER		ONE
THE	BOY	WHO	LIVED
r.	and	Mrs.	Dursley,	of	number	four,	Privet	Drive,	were	proud	to	say
that	they	were	perfectly	normal,	thank	you	very	much.	They	were	the
last	people	you’d	expect	to	be	involved	in	anything	strange	or	mysterious,
because	they	just	didn’t	hold	with	such	nonsense.
Mr.	Dursley	was	the	director	of	a	firm	called	Grunnings,	which	made	drills.
He	was	a	big,	beefy	man	with	hardly	any	neck,	although	he	did	have	a	very
large	mustache.	Mrs.	Dursley	was	thin	and	blonde	and	


In [6]:
class InputOuptputDataset(Dataset):
    def __init__(self,txt,context_length,tokenizer,stride):
      self.txt=txt
      self.context_length=context_length
      self.tokenizer=tokenizer
      self.stride = stride

      self.input=[]
      self.output=[]

      encoded_ids=tokenizer.encode(txt,allowed_special={"<|endoftext|>"})
      for i in range(0,(len(encoded_ids)-context_length),stride):
        self.input.append(torch.tensor(encoded_ids[i:i+context_length]))
        self.output.append(torch.tensor(encoded_ids[i+1:i+context_length+1]))



    def __len__(self):
      return len(self.input)

    def __getitem__(self,idx):
      context=self.input[idx]
      target=self.output[idx]
      return context,target


In [7]:
def create_dataloader(txt,batch_size,shuffle=True,max_length=256,stride=128,drop_last=True,num_workers=0):
  tokenizer=tiktoken.get_encoding('gpt2')
  dataset = InputOuptputDataset(txt,max_length,tokenizer,stride)
  dataloader_init = DataLoader(dataset,batch_size=batch_size,shuffle=shuffle,drop_last=drop_last,num_workers=num_workers)
  return dataloader_init



In [19]:
dataloader = create_dataloader(pdf_text,batch_size=8,shuffle=False,max_length=4,stride=4,drop_last=True,num_workers=0)

data_iter = iter(dataloader)
inputs,targets = next(data_iter)
print(inputs)

tensor([[   44,   198,   197,   198],
        [41481,   197,   197, 11651],
        [  198, 10970,   197,  8202],
        [   56,   197, 41856,   197],
        [   43,  3824,  1961,   198],
        [   81,    13,   197,   392],
        [  197, 27034,    13,   197],
        [   35,  1834,  1636,    11]])


In [23]:
vocabulary_size= 50276
output_dim = 256

In [24]:
embedding = nn.Embedding(vocabulary_size,output_dim)

In [33]:
token_embeddings=embedding(inputs)
token_embeddings.shape

torch.Size([8, 4, 256])

In [34]:
max_length=context_length=4
output=output_dim

In [35]:
positional_embedding = nn.Embedding(max_length,output)
pos_embeddings=positional_embedding(torch.arange(max_length))
pos_embeddings.shape

torch.Size([4, 256])

In [36]:
input_embeddings = token_embeddings + pos_embeddings
print(input_embeddings.shape)

torch.Size([8, 4, 256])


So what is important here is to understand the dimensions of the vectors
1. our dataloader context length is 4 batch size is 8 this means each input vector has 4 tokens and  and each batch has 8 such input vectors ie dim is 8*4
2. dimension of embedding is vocab_size*output_dim
ex: 50276*256
3. now each token would be projected to dimension of 256
hence after embedding the dimension is 8*4*256
4. now positional vectors is addede to each token means if we have 4 tokens then 4 positional vector= the context length  so dimension is 4*256
. at the end the dimension is  8*4*256 when token embedding and positional embedding is added